Rapport TP comptage des mots
Johan Balduzzi INFO FISE 5A

Partie 1 : Python simple
L'objectif de cette partie est de compter le nombre d'occurence des mots dans le fichier mots.txt en utilisant un dictionnaire python.

On commence par générer le fichier mots.txt avec une liste de mots prédéfinis.

In [11]:
import random

comptes = {"chat":0,"chien":0,"arbre":0,"fleur":0,"maison":0,"soleil":0,
           "montagne":0,"livre":0,"ciel":0,"mer":0,"lune":0,"route":0,
           "oiseau":0,"jardin":0,"voiture":0}

for k in comptes.keys():
    comptes[k] = random.randint(1, 10)

mots = [m for m, n in comptes.items() for _ in range(n)]
random.seed(42)
random.shuffle(mots)

with open("mots.txt", "w", encoding="utf-8") as f:
    for i in range(0, len(mots)):
        f.write(mots[i] + "\n")

print("fichier crée")

fichier crée


On va maintenant compter le nombre d'occurence de chaque mots

In [69]:
%%time
def compter_occurences(fichier):
    with open(fichier, 'r', encoding='utf-8') as f:
        mots = f.read().split()

    total_mot = 0
    compteur = {}
    for mot in mots:
        compteur[mot] = compteur.get(mot, 0) + 1
        total_mot += 1
    print("total = ", total_mot)
    return compteur

if __name__ == "__main__":
    fichier = "mots.txt"
    resultats = compter_occurences(fichier)
    for mot, nb in sorted(resultats.items(), key=lambda x: x[1], reverse=True):
        print(f"{mot} : {nb}")
    

total =  784
jardin : 80
chien : 80
route : 80
montagne : 80
arbre : 72
maison : 64
soleil : 56
chat : 48
lune : 48
livre : 48
ciel : 32
voiture : 32
mer : 24
oiseau : 24
fleur : 16
CPU times: user 731 μs, sys: 3.3 ms, total: 4.03 ms
Wall time: 2.99 ms


Partie 2 Pseudo-MapReduce 
L'objectif de cette partie est de reproduire les étapes du modèle MapReduce à l'aide de fonctions mapper, shuffle et reducer.

In [72]:
%%time

from collections import defaultdict

def mapper(ligne):
    mots = ligne.strip().split()
    return [(mot, 1) for mot in mots]

def shuffle(mapped_data):
    grouped = defaultdict(list)
    for mot, valeur in mapped_data:
        grouped[mot].append(valeur)
    return grouped

def reducer(grouped_data):
    reduced = {}
    for mot, valeurs in grouped_data.items():
        reduced[mot] = sum(valeurs)
    return reduced

if __name__ == "__main__":
    fichier = "mots.txt"
    mapped = []
    with open(fichier, "r", encoding="utf-8") as f:
        for ligne in f: 
            mapped.extend(mapper(ligne))
    grouped = shuffle(mapped)
    reduced = reducer(grouped)

    for mot, count in sorted(reduced.items(), key=lambda x: x[1], reverse=True):
        print(f"{mot} : {count}")    
    

jardin : 80
chien : 80
route : 80
montagne : 80
arbre : 72
maison : 64
soleil : 56
chat : 48
lune : 48
livre : 48
ciel : 32
voiture : 32
mer : 24
oiseau : 24
fleur : 16
CPU times: user 3.48 ms, sys: 0 ns, total: 3.48 ms
Wall time: 3 ms


Partie 3 : Hadoop
L'objectif de cette partie est d'exécuter le WordCount sur HDFS en utilisant Hadoop

In [79]:
%%time

!hdfs dfs -mkdir -p /user/$USER/wordcount/input
!hdfs dfs -put -f mots.txt /user/$USER/wordcount/input/

!hadoop jar $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar wordcount /user/$USER/wordcount/input /user/$USER/wordcount/output

!hdfs dfs -cat /user/$USER/wordcount/output/part-r-00000

2026-10-13 15:28:01,434 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
2026-10-13 15:28:01,742 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/johan/.staging/job_1791884691952_0015
2026-10-13 15:28:02,034 INFO input.FileInputFormat: Total input files to process : 1
2026-10-13 15:28:02,119 INFO mapreduce.JobSubmitter: number of splits:1
2026-10-13 15:28:02,290 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1791884691952_0015
2026-10-13 15:28:02,290 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-10-13 15:28:02,455 INFO conf.Configuration: resource-types.xml not found
2026-10-13 15:28:02,455 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-10-13 15:28:02,526 INFO impl.YarnClientImpl: Submitted application application_1791884691952_0015
2026-10-13 15:28:02,555 INFO mapreduce.Job: The url to track the job: http://DELL-Johan.localdomain:8088/proxy/ap

Suppression de l'ancien output pour mettre à jour les futurs résultats.

In [80]:
!hdfs dfs -rm -r -f /user/$USER/wordcount/output

Deleted /user/johan/wordcount/output


Premiers Résultats : Tests sur 3 essais

Le comptage de mots en python simple à pris (1.37ms, 2.73ms, 4.17ms) ce qui est très rapide pour un comptage manuel car le fichier est petit.

La version pseudo-MapReduce à pris (1.31ms, 2.48ms, 8.01ms). Les résultats sont assez proches de la méthodes simple avec un résultats assez élevé.

La méthode avec Hadoop à pris (1.31s, 1.44s, 1.45s) ce qui est assez élevé. Le résultat est différent du temps d'exécution de la cellule puisque je supprime l'output pour le reconstruire à chaque fois. Je prend en compte le résultat obtenu avec %%time.

Nous allons maintenant augmenter la taille du fichier mots.txt. Pour cela, nous allons doubler le nombre d'occurence de chaques mots présents dans ce fichier.

In [65]:
with open("mots.txt", "r", encoding="utf-8") as f:
    contenu = f.read()

with open("mots.txt", "w", encoding="utf-8") as f:
    f.write(contenu)
    if not contenu.endswith("\n"):
        f.write("\n")
    f.write(contenu)

On renvoi le fichier sur HDFS pour que Hadoop traite la bonne version.

In [66]:
!hdfs dfs -put -f mots.txt /user/$USER/wordcount/input/

Second résultats : 

La version de python simple a un temps d'exécution de   (8.16ms, 2.25ms, 5.49ms). Les temps d'exécution sont supérieurs aux premiers tests avec un résultat anormal de 8.16ms.

La version pseudo-MapReduce a un temps d'exécution de  (1.71ms, 1.32ms, 2.99ms) ce qui est relativement proche des résultats des premiers tests si on ne compte pas le résultat anormal de 8.01ms du premier test.

La méthode Hadoop a un temps d'exécution de (1.26s, 1.42s, 1.35s) ce qui est similaire aux premiers tests. On peut faire l'hypotèse que sur un fichier de petite taille, le temps d'exécution ne varie pas ou peu en prenant en compte que hadoop est principalement utilisé sur de gros volumes de données. 

Partie 4 : Utilisation de spark
L'objectif de cette partie est de procéder au comptage de mot en utilisant l'outil spark.
On commence par installer le module.

In [14]:
%pip install "pyspark==3.5.3"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.3/317.3 MB 5.0 MB/s eta 0:00:0000:0100:02
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyspark: filename=pyspark-3.5.3-py2.py3-none-any.whl size=317840689 sha256=b83be9903d3c44d21ecbc84efbc66871ddac82594adb5f8c6cfcb4d512203662
  Stored in directory: /home/johan/.cache/pip/wheels/07/a0/a3/d24c94bf043ab5c7e38c30491199a2a11fef8d2584e6df7fb7
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.2.0
    Uninstalling pyspark-4.2.0:
      Successfully uninstalled pyspark-4.2.0

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use upda

On implémente maintenant une fonction pour compter le nombre de mots.

In [83]:
%%time

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("WordCount") \
    .master("local[*]") \
    .getOrCreate()
sc = spark.sparkContext

def spark_count():
    rdd = sc.textFile("mots.txt")
    return (rdd.flatMap(lambda ligne: ligne.split())
                .map(lambda mot: (mot, 1))
                .reduceByKey(lambda a, b: a + b)
                .collect()
            )

resultats = spark_count()
for mot, nb in sorted(resultats, key=lambda x: x[1], reverse=True):
    print(f"{mot} : {nb}")

jardin : 80
route : 80
chien : 80
montagne : 80
arbre : 72
maison : 64
soleil : 56
lune : 48
livre : 48
chat : 48
ciel : 32
voiture : 32
oiseau : 24
mer : 24
fleur : 16
CPU times: user 23.9 ms, sys: 2.04 ms, total: 25.9 ms
Wall time: 359 ms


Le comptage de mot avec Spark a un temps d'exécution de (33.1ms, 36.1ms, 29.1ms) dans la deuxième version du fichier mots.txt (le premier doublage de la taille). Ces temps sont plus longs que ceux de python simple et de pseudo-MapReduce. En se basant sur le même principe que hadoop, les performances de Spark seront révélé lorsque la taille du fichier augmentera.

Pour plus de précision pour comparer les méthodes, nous allons à nouveau doubler la taille du fichier.

Résultats 3 : 

Le temps d'exécution de la méthode pyhton simple est de (2.54ms, 4.08ms, 3.34ms). Hors résultats aberrant, le temps moyen augmente par rapport aux tests précédents ce qui est cohérent avec l'augmentation de la taille des données.

Le temps d'exécution de la méthode pseudo-MapReduce est de (3.23ms, 3.49ms, 7.17ms). Le temps moyen est également supérieurs aux autres tests si on enlève le résultat anormal de 7.17ms.

Le temps d'exécution de la méthode hadoop est de (1.42s, 1.22s, 1.34s) ce qui est similaire aux tests précédents. L'augmentation du volume de données est encore insuffisant pour mesurer précisément les performances de hadoop.

Le temps d'exécution de Spark est de (39.6ms, 29.1ms, 25.1ms) ce qui est également similaire aux tests précédents. La taille des données est insuffisant pour mesurer les performances de Spark.

Nous allons continuer à doubler la taille des données. 
Les résultats seront directement notés dans un tableau sans analyse individuelle.

Voici des tableaux récapitulatif des différents résulatas obtenus : 
version 1 = simple
version 2 = pseudo-MapReduce
version 3 = Hadoop
version 4 = Spark

Taille Facteur 1 : 

        version 1 | version 2 | version 3 | version 4
essai 1| 1.37ms   | 1.31ms    | 1.31s     |
essai 2| 2.73ms   | 2.48ms    | 1.44s     |
essai 3| 4.17ms   | 8.01ms    | 1.45s     |

Taille Facteur 2 : 

        version 1 | version 2 | version 3 | version 4
essai 1| 8.16ms   | 1.71ms    | 1.26s     | 33.4ms
essai 2| 2.25ms   | 1.32ms    | 1.42s     | 36.1ms
essai 3| 5.49ms   | 2.99ms    | 1.35s     | 29.1ms

Taille Facteur 4 : 

        version 1 | version 2 | version 3 | version 4
essai 1| 2.54ms   | 3.23ms    | 1.42s     | 39.6ms
essai 2| 4.08ms   | 3.49ms    | 1.22s     | 29.1ms
essai 3| 3.34ms   | 7.17ms    | 1.34s     | 25.1ms

Taille Facteur 8 : 

        version 1 | version 2 | version 3 | version 4
essai 1| 2.25ms   | 3.53ms    | 1.34s     | 33.9ms
essai 2| 3.18ms   | 5.19ms    | 1.24s     | 31.8ms
essai 3| 4.03ms   | 3.48ms    | 1.23s     | 25.9ms